Needs a lot of editing and debugging but will serve as a protocol for analysing RECTIFY images 

# Step 0: Preparation

In [16]:
import torch
import numpy as np #for running adc quantification in the notebook
%gui qt

Choose where to take scans from and where to output the results

In [7]:
DATAPATH = r"C:\Users\cmq26nc\neoadj\venvs\ppln-rectify\scans\Patient_01" #change RECTIFY01 to RECTIFY_XY for each patient
BUILD_PATH = r"C:\Users\cmq26nc\neoadj\venvs\rectify\results\Patient_01_13" #need to change this to a new folder for each run

import os

IMAGEPATH = os.path.join(BUILD_PATH, 'images')
CHECKPOINTSPATH = os.path.join(BUILD_PATH, 'checkpoints')
os.makedirs(IMAGEPATH, exist_ok=True)
os.makedirs(CHECKPOINTSPATH, exist_ok=True)

resultspath = os.path.join(BUILD_PATH, 'dicom')
SUBJ = 'SheffieldSubject'
DATA = [resultspath, SUBJ, 'Data']
MAPS = [resultspath, SUBJ, 'Maps']
MASKS = [resultspath, SUBJ, 'Masks']

Check environment path to ensure smooth operation

In [8]:
import sys
from pathlib import Path

print(sys.executable)
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the project root (pyproject.toml) from the notebook directory.")
source_path = str(project_root / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)


c:\Users\cmq26nc\AppData\Local\Programs\Python\Python313\python.exe


Display file names

In [9]:
import dbdicom as db
for series in db.series(DATAPATH):
    print(series)
#does each new run make a copy of perfusion (flip 2)?
#no it just seems to be a single copy

['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('3-Plane Localizer', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('SAG SSFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('COR SSFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('Ax SSFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('Sag T2 frFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('Ax T2 frFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('POLARIS_RECTIFY', 0), ('Cor T2 frFSE', 0)]
['C:\\Users\\cmq26nc\\neoadj\\venvs\\ppln-rectify\\scans\\Patient_01', 'RECTIFY_001', ('P

# Step 1: Data cleaning

Harmonize the DICOM data, saving the results in the build folder

Similar procedure to lung case

In [10]:
from rectify.rectify_code import harmonize_data

harmonize_data(DATAPATH, DATA)
#runs in about 15-20 minutes (hardware dependent, all estimates in this notebook are based on intel core i7 desktop)
#added lines in rectify_code.py to give both the dicom SeriesDescription (i.e. what is shown in the copying series sections) but also the source folder
#done to check it would still work with minor file naming incosistencies e.g flip_2_ vs flip4_ 

Reading DICOM folder: 0it [00:00, ?it/s]


Merging SeriesDescription: PERFUSION (flip 2); source folder: 11-PERFUSION__flip_2_


Copying series ['SheffieldSubject', 'Data', '3D-VFA-2']: 100%|██████████| 50/50 [00:00<00:00, 55.90it/s]
Sorting series..: 100%|██████████| 50/50 [00:00<00:00, 260.77it/s]
Writing values..: 50it [00:00, 52.20it/s]
Copying series ['SheffieldSubject', 'Data', '3D-VFA']: 100%|██████████| 50/50 [00:00<00:00, 52.23it/s]


Merging SeriesDescription: PERFUSION (flip 4); source folder: 10-PERFUSION__flip4_


Copying series ['SheffieldSubject', 'Data', '3D-VFA-4']: 100%|██████████| 50/50 [00:01<00:00, 46.82it/s]
Sorting series..: 100%|██████████| 50/50 [00:00<00:00, 234.19it/s]
Writing values..: 50it [00:01, 39.39it/s]
Copying series ['SheffieldSubject', 'Data', '3D-VFA']: 100%|██████████| 50/50 [00:00<00:00, 52.69it/s]


...

Merging SeriesDescription: PERFUSION (flip 30); source folder: 14-PERFUSION__flip_30_


Copying series ['SheffieldSubject', 'Data', '3D-VFA-30']: 100%|██████████| 50/50 [00:00<00:00, 62.28it/s]
Sorting series..: 100%|██████████| 50/50 [00:00<00:00, 256.72it/s]
Writing values..: 50it [00:01, 47.62it/s]
Copying series ['SheffieldSubject', 'Data', '3D-VFA']: 100%|██████████| 50/50 [00:01<00:00, 42.86it/s]


Merging SeriesDescription: PERFUSION DISCO; source folder: 15-PERFUSION_DISCO


Copying series ['SheffieldSubject', 'Data', '3D-DCE']: 100%|██████████| 15000/15000 [03:40<00:00, 68.05it/s]
Sorting series..: 100%|██████████| 15000/15000 [00:42<00:00, 352.15it/s]
Writing values..: 15000it [04:03, 61.60it/s]


Finished harmonizing data. VFA and DCE series are now merged into single series for analysis.


In [6]:
from rectify.rectify_code import view_dce_notebook

view_dce_notebook(DATA)
# ~2.5 mins it will keep running after the volume is 100% read just give it a minute and napari will open. 
# #Close napari to continue running the notebook if run all was selected.

Reading volume..: 100%|██████████| 15000/15000 [01:32<00:00, 162.17it/s]


# Step 2: Map Computaation

Compute some descriptive maps that can be used as a basis for segmentation and eventually calculate T1 and S0

ADC maps calculated using DWI images (just axial for patient 1)

In [12]:
from rectify.rectify_code import compute_descriptive_maps

compute_descriptive_maps(DATA, MAPS)
# ~3mins

Reading volume..: 100%|██████████| 15000/15000 [00:58<00:00, 255.43it/s]
Writing volume..: 50it [00:00, 58.69it/s]
Writing volume..: 50it [00:00, 73.34it/s]
Writing volume..: 50it [00:00, 72.73it/s]
Writing volume..: 50it [00:00, 67.93it/s]
Writing volume..: 50it [00:00, 73.49it/s]


Finished computing descriptive maps. Maps are saved in the MAPS series.


Create an ADC map from the axial DWI series with T2 weighted images, will add coronal and saggital if later cases have DWI in those planes

b-values taken from DICOM folders

In [13]:
from rectify.rectify_code import adc_map

adc_map(DATAPATH, MAPS)

Finished creating ADC map in native axial DWI space (units: mm^2/s).


In [14]:
from rectify.rectify_code import view_adc_notebook

view_adc_notebook(MAPS)
#  bright spots at tumour sites due to limited browmian motion in tumour volume

Reading volume..: 100%|██████████| 25/25 [00:00<00:00, 152.09it/s]


Viewer(camera=Camera(center=(0.0, np.float64(119.53125), np.float64(119.53125)), zoom=np.float64(2.375), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(55.19999885559082), 1.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=2, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(110.39999771118164), step=np.float64(4.599999904632568)), RangeTuple(start=np.float64(0.0), stop=np.float64(239.0625), step=np.float64(0.9375)), RangeTuple(start=np.float64(0.0), stop=np.float64(239.0625), step=np.float64(0.9375))), margin_left=(0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0), point=(np.float64(55.19999885559082), np.float64(119.0625), np.float64(119.062

In [36]:
adc = db.volume(MAPS + ['ADC']).values
valid = np.isfinite(adc) & (adc > 0)
if not np.any(valid):
    raise ValueError("The ADC map has no positive finite values.")
print(f"Maximum ADC: {(np.max(adc[valid])):.5g} mm^2/s")
print(f"Top 1% ADC Values: {np.percentile(adc[valid], 99):.5g} mm^2/s")
print(f"Top 10% ADC Values: {np.percentile(adc[valid], 90):.5g}mm^2/s")
print(f"Mean ADC Value: {np.percentile(adc[valid], 50):.5g} mm^2/s") #(ADC of rectal wall typically 0.0018-0.001405)
print(f"Minimum ADC: {(np.min(adc[valid])):.5g}  mm^2/s")

Reading volume..:   0%|          | 0/25 [00:00<?, ?it/s]

Reading volume..: 100%|██████████| 25/25 [00:00<00:00, 260.98it/s]


Maximum ADC: 0.0051175 mm^2/s
Top 1% ADC Values: 0.0035797 mm^2/s
Top 10% ADC Values: 0.0020856mm^2/s
Mean ADC Value: 0.00090482 mm^2/s
Minimum ADC: 5.6815e-08  mm^2/s


# Step 3: Image Segmentation

Colon can be automatic but rectum will have to be done manually

In [10]:
from rectify.rectify_code import autosegment_colorectal

autosegment_colorectal(MAPS, MASKS)
# ~5 mins 
#this is the very loud step

Reading volume..: 100%|██████████| 50/50 [00:00<00:00, 89.70it/s]


Saving source as nifti..
Segmenting organs..

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 1.29s
Predicting part 1 of 2 ...


c:\Users\cmq26nc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\jit\_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
  0%|          | 0/9 [00:11<?, ?it/s]


KeyboardInterrupt: 

Check the masks and edit interactively, if needed. Almost certainly needed for rectum due to lack of total_mr map

In [ ]:
import inspect
import rectify.rectify_code as rc

print(rc.__file__)
print(inspect.getsource(rc.edit_autosegmented_colorectal))

c:\Users\cmq26nc\neoadj\venvs\ppln-rectify\src\rectify\rectify_code.py
def edit_autosegmented_colorectal(maps, masks):

    # Edit colorectal on baseline image
    ref_series = maps + ['Sb']
    ref_vol = db.volume(ref_series) 

    # Edit  ROIs
    masks_to_edit = ['colon']
    for mask in masks_to_edit:
        mask_array = db.volume(masks + [mask]).values
        mask_array = draw_mask(ref_vol.values, mask=mask_array, name=mask)
        mask_vol = (mask_array, ref_vol.affine)
        mask_series = masks + [f"{mask}_edited"]
        db.delete(mask_series, not_exists_ok=True)
        db.write_volume(mask_vol, mask_series, ref=ref_series)

    print('Finished editing colon segmentations. Edited masks are saved in the MASKS series.')



In [ ]:
from rectify.rectify_code import edit_autosegmented_colorectal

edit_autosegmented_colorectal(MAPS, MASKS)

#It seems like all segmentation in this project will need to be manually done, look into methods for this.

Reading volume..:   0%|          | 0/50 [00:00<?, ?it/s]

Reading volume..: 100%|██████████| 50/50 [00:00<00:00, 192.07it/s]


ValueError: Series colon not found in study Masks.

In [ ]:
from  rectify.rectify_code import draw_arterial_inputs_notebook

draw_arterial_inputs_notebook(MAPS, MASKS)
# Close when finished to continue

Reading volume..: 100%|██████████| 50/50 [00:00<00:00, 169.68it/s]
Writing volume..: 50it [00:01, 37.66it/s]


# Step 4: Breathing motion correction

Motion-correct the VFA data for signal calibration

In [ ]:
from rectify.rectify_code import vfa_motion_correction

vfa_motion_correction(DATA, MAPS)
# also very loud at the start and when fitting deformation fields
# ~2 hours

Reading volume..: 100%|██████████| 250/250 [00:02<00:00, 114.86it/s]


Initializing..
Iteration 1: fitting signal model


Fitting VFA model: 100%|██████████| 13107200/13107200 [16:25<00:00, 13301.42it/s]
c:\Users\cmq26nc\AppData\Local\Programs\Python\Python313\Lib\site-packages\mdreg\fit_models.py:918: RuntimeWarning: divide by zero encountered in divide
  E = 1/A


Iteration 1: fitting deformation fields
Calculation time for iteration 1: 22.25324569940567 min
Iteration 2: fitting signal model


Fitting VFA model: 100%|██████████| 13107200/13107200 [16:48<00:00, 12992.85it/s]


...

Fitting VFA model: 100%|██████████| 13107200/13107200 [16:02<00:00, 13617.63it/s]


Iteration 4: fitting deformation fields
Calculation time for iteration 4: 21.012342103322347 min
Iteration 5: fitting signal model


Fitting VFA model: 100%|██████████| 13107200/13107200 [15:29<00:00, 14096.82it/s]


Iteration 5: fitting deformation fields
Calculation time for iteration 5: 21.36571201086044 min
Total calculation time: 110.55543161630631 min


Writing volume..: 100%|██████████| 5/5 [00:11<00:00,  2.37s/it]


Check the motion-corrected VFA data

In [ ]:
from rectify.rectify_code import view_vfa_notebook

view_vfa_notebook(MAPS)
# Gives a pretty good output

Reading volume..: 100%|██████████| 250/250 [00:02<00:00, 124.25it/s]


Perform DCE motion correction

*Note* this is actually a dummy function for this demo which just creates a copy. Motion correction takes a long time and these colorectal scans don't really need it as the abdominal motion is minimal, so in this case it can be safely skipped.

In [ ]:
from rectify.rectify_code import dce_motion_correction

dce_motion_correction(DATA, MAPS)

Writing volume..: 100%|██████████| 300/300 [06:15<00:00,  1.25s/it]


# Step 5: Signal calibration

Fit the VFA data to derive T1 and S0 maps


In [12]:
#Would it be beneficial to merge perfusion DISCO with the merged vfa perfusion images? I'm assuming disco is cfa so might make less sense to merge but i'm not sure if that could lead to useful information

In [ ]:
from rectify.rectify_code import vfa_fit

vfa_fit(MAPS)
#kernel crashed halfway through check if this will happen again, have to restart the script and see if it happens
#kernel also sometimes dies in earlier steps, is this due to very large number of images?

ModuleNotFoundError: No module named 'rectify'

Align VFA and DCE data, while computing scaling factors for the MR signal intensities (arbitrary units).

In [ ]:
from rectify.rectify_code import align_vfa_with_dce

align_vfa_with_dce(MAPS, MASKS)

# Step 6: Parameter reconstruction

Now we have everything in place to model the data and derive the parameters of interest.

Model-based colon analysis

In [ ]:
from rectify.rectify_code import roi_analysis_model_based

roi_analysis_model_based(MAPS, MASKS, IMAGEPATH)

Reading volume..: 100%|██████████| 15000/15000 [01:32<00:00, 161.50it/s]


ValueError: Study Masks not found in patient SheffieldSubject.

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'model_based_colon_edited_rectify01.png')
view_png(png)


Model-free colon analysis

In [ ]:
from rectify.rectify_code import roi_analysis_model_free

roi_analysis_model_free(MAPS, MASKS, IMAGEPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'model_free_colon_edited_rectify01.png')
view_png(png)


Pixel-by-pixel model-free analysis

In [ ]:
from rectify.rectify_code import pixel_analysis_model_free

pixel_analysis_model_free(MAPS, MASKS, IMAGEPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'Fb_model_free_rectify01.png')
view_png(png)
png = os.path.join(IMAGEPATH, f'vb_model_free_rectify01.png')
view_png(png)

Pixel-by-pixel model-based analysis

In [ ]:
from rectify.rectify_code import pixel_analysis_model_based

pixel_analysis_model_based(MAPS, MASKS, IMAGEPATH, CHECKPOINTSPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'Fb_model_based_rectify01.png')
view_png(png)
png = os.path.join(IMAGEPATH, f'vb_model_based_rectify01.png')
view_png(png)

# Step 7: Export imaging biomarkers

In [ ]:
from rectify.rectify_code import pixel_averages_model_based

pixel_averages_model_based(MAPS, MASKS, SUBJ, BUILD_PATH)